# 02 — Assertions, evidence, and a bounded biological question

We ask: **which diseases are connected to TP53 in this bounded graph, and what source records support each assertion?** DuckDB performs the join without loading a full KG into Python memory.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "public":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


In [ ]:
root = str(KG_ROOT).rstrip("/")
edges = read_bounded_parquet(f"{root}/edges/disease_associated_gene.parquet", limit=20, billing_project=BILLING_PROJECT)
evidence = read_bounded_parquet(f"{root}/evidence/disease_associated_gene.parquet", limit=20, billing_project=BILLING_PROJECT)
display(edges)
display(evidence[["edge_key", "source", "source_dataset", "evidence_score", "predicate"]])


In [ ]:
from manage_db.public_notebooks import diseases_with_gene_evidence

if MODE != "fixture":
    print("The reusable DuckDB question helper currently requires local bounded Parquets. Stage only the three selected objects, then pass that local root.")
else:
    answer = diseases_with_gene_evidence(KG_ROOT, "ENSG00000141510", limit=20)
    display(answer)


## What this means — and does not mean

The result shows represented associations and their attached provenance. Scores are source-specific and are not calibrated probabilities of causality. Association direction, tissue context, ancestry, assay type, and publication bias must be checked in evidence metadata before scientific interpretation. Do not infer treatment efficacy from a gene–disease neighborhood.
